In [ ]:
# Resolve defaults from this project's root, independently of the working directory.
from pathlib import Path as _KSPath
import sys as _ks_sys
from functools import partial as _ks_partial
_ks_start = _KSPath.cwd()
_ks_root = next((p for p in (_ks_start, *_ks_start.parents) if (p / 'kinspec_paths.py').is_file()), None)
if _ks_root is None:
    raise RuntimeError('Cannot locate the KinSPEC project root containing kinspec_paths.py. Start the notebook kernel from the project directory or a subdirectory.')
_ks_sys.path.insert(0, str(_ks_root))
from kinspec_paths import ROOT as KINSPEC_ROOT, project_path as _ks_project_path
_kp = _ks_partial(_ks_project_path, script_dir='model_training')
import os as _ks_os
_ks_os.chdir(KINSPEC_ROOT / 'model_training')



In [ ]:
import os
import hashlib
import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.metrics import precision_recall_curve,log_loss,accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,matthews_corrcoef,roc_auc_score
import optuna
import gc
from scipy.stats import norm
from collections import Counter
from scipy.stats import norm
from functools import partial
from sklearn.model_selection import StratifiedKFold

In [ ]:
# Bin bioactivity values into binary labels
def bio_cal(bioactivity):
    # Clip PKIS1 values because some entries are above 100 or below 0
    bioactivity = bioactivity.clip(0,100)
    # Binning rule: define fixed intervals and their corresponding labels
    bins = [-np.inf, 50, np.inf]  # Intervals used to classify the normalized bioactivity values
    labels = [0, 1]  # Binary labels for activity strength
    
    final_bioactivity = pd.cut(
    bioactivity, 
    bins=bins, 
    labels=labels, 
    include_lowest=True
    )
    return final_bioactivity

In [ ]:
MODEL_PATH = _kp('model/PSM_raw/KinSPEC_LTS_Karaman_sim05_training_pool_pure_PSM_v4.json')

# import training data
# Read only the label and sequence columns; avoid loading SMILES, Morgan and source.
warm_train = pd.read_csv(_kp('dataset/KinSPEC_LTS_merged_4datasets_deduplicated_no_karaman_sim0.5.csv'),usecols=["KLIFS", "unified_bio_label"])

# import mol LSAE embedded features
# float32 halves the footprint and is the precision XGBoost works in internally anyway.
warm_train_mol_fp = pd.read_csv(_kp('dataset/KinSPEC_LTS_Karaman_sim0.5_SAE_latent_top95_features_from_merge_v4.csv'),dtype=np.float32)

# import bioactivity classification from data
warm_bio = warm_train["unified_bio_label"]

# mapping KLIFS to pure KLIFS PSM feature
# Held as float32 instead of int64: the one-hot table is 0/1, so int64 wasted 8 bytes per bit.
KLIFS_sae_top = pd.read_csv(_kp('feature_embedding/Kinase_test_MCA/human_482_typical_kinase_1955bit_onehot.csv'),
    header=None, dtype=np.float32).to_numpy(copy=False)
KLIFS_file = pd.read_csv(_kp('feature_embedding/Kinase_test_MCA/human_482_typical_kinase_1955bit.csv'))

In [ ]:
def klifs_index(df, KLIFS_file):

    ref = KLIFS_file["KLIFS"].astype(str).str.strip()
    # `dict(zip(keys, values))` silently kept the LAST of any duplicated KLIFS.
    # groupby(level=0).last() reproduces that exactly. (The reference table has one
    # duplicated sequence and both of its one-hot rows are identical, so this is a no-op.)
    lookup = pd.Series(np.arange(len(ref)), index=ref).groupby(level=0).last()

    pos = df["KLIFS"].astype(str).str.strip().map(lookup)
    n_missing = int(pos.isna().sum())
    if n_missing:
        raise ValueError(f"{n_missing} KLIFS sequences were not found in the reference table")
    return pos.to_numpy(dtype=np.intp)


def build_features(df, mol_fp, KLIFS_file, KLIFS_latent, chunk=50_000):

    pos = klifs_index(df, KLIFS_file)
    n, m = mol_fp.shape
    assert n == len(pos), "The number of feature rows does not match the number of KLIFS rows!"

    X = np.empty((n, m + KLIFS_latent.shape[1]), dtype=np.float32)
    X[:, :m] = mol_fp.to_numpy(dtype=np.float32, copy=False)
    for start in range(0, n, chunk):
        stop = min(start + chunk, n)
        np.take(KLIFS_latent, pos[start:stop], axis=0, out=X[start:stop, m:])
    return X

In [ ]:
# Concatenate molecular latent features and kinase latent features
X_train = build_features(warm_train, warm_train_mol_fp, KLIFS_file, KLIFS_sae_top)
y_train = warm_bio.to_numpy()
assert len(X_train) == len(y_train), "Feature rows and labels must match"
N_FEATURES = X_train.shape[1]


In [ ]:
# check shape
print("train rows           :", len(warm_train))
print("mol latent features  :", warm_train_mol_fp.shape[1])
print("kinase one-hot bits  :", KLIFS_sae_top.shape[1])
print("X_train              :", X_train.shape, X_train.dtype, f"({X_train.nbytes / 1e9:.2f} GB)")
print("y_train              :", y_train.shape, y_train.dtype)
print("label distribution   :", np.bincount(y_train), "(inactive, active)")

In [ ]:
# Delete intermediate variables to reduce memory usage
del warm_train_mol_fp, warm_train, warm_bio
# Run garbage collection
gc.collect()

In [ ]:
N_FOLDS = 5

# Build the cross-validation split ONCE and reuse it for every trial.
# StratifiedKFold's split depends only on the label values, their row order and random_state --
# not on X's dtype or container -- so these are the same folds the original code produced with
# StratifiedKFold(n_splits=5, shuffle=True, random_state=42) inside `objective`.
FOLDS = list(
    StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42).split(X_train, y_train)
)

# Fingerprint of the concatenated validation indices. This pins the exact split, so any future
# change that perturbs the folds fails here instead of silently shifting the reported scores.
EXPECTED_FOLD_FINGERPRINT = "b2eea7bd77cbcfc113549b75711744db0639ff3398947294c59bfe7d51377c62"
fold_fingerprint = hashlib.sha256(
    b"".join(valid.astype(np.int64).tobytes() for _, valid in FOLDS)
).hexdigest()
print("fold fingerprint:", fold_fingerprint)
assert fold_fingerprint == EXPECTED_FOLD_FINGERPRINT, "the cross-validation split has changed!"

for fold_id, (train_idx, valid_idx) in enumerate(FOLDS, start=1):
    print(
        f"  fold{fold_id}: train={len(train_idx)} valid={len(valid_idx)} "
        f"valid_positives={int(y_train[valid_idx].sum())}"
    )

In [ ]:
def make_fold_cache(X, y, folds):

    cache = []
    for train_idx, valid_idx in folds:
        dtrain = xgb.QuantileDMatrix(X[train_idx], label=y[train_idx])
        dvalid = xgb.QuantileDMatrix(X[valid_idx], label=y[valid_idx], ref=dtrain)
        cache.append((dtrain, dvalid, y[valid_idx]))
        gc.collect()  # release the fancy-index copy before slicing the next fold
    return cache


# Build the quantized folds once, avoiding DMatrix slices in every trial.
FOLD_CACHE = make_fold_cache(X_train, y_train, FOLDS)
print(f"cached {len(FOLD_CACHE)} folds:",
      [(d.num_row(), v.num_row()) for d, v, _ in FOLD_CACHE])

In [ ]:
# set bedroc
def calculate_bedroc(y_true, y_score, bedroc_alpha):
    # y_true: true binary labels
    y_true  = np.asarray(y_true)
    # y_score: predicted scores
    y_score = np.asarray(y_score)
    N = len(y_true)
    n_act = np.sum(y_true == 1)
    if n_act == 0:
        return 0.0                    # Return 0 when there are no active compounds

    # 1. Sort predictions and extract the ranks of active compounds
    order  = np.argsort(y_score)[::-1]
    ranks  = np.arange(1, N+1)[y_true[order] == 1]      # 1-based
    x      = (ranks - 0.5) / N                          # Centered rank

    # 2. Calculate RIE
    C      = (1.0 - np.exp(-bedroc_alpha)) / bedroc_alpha
    rie    = np.mean(np.exp(-bedroc_alpha * x)) / C

    # 3. Calculate the theoretical limits
    x_max  = (np.arange(1, n_act+1) - 0.5) / N          # Best case
    x_min  = (np.arange(N-n_act+1, N+1) - 0.5) / N      # Worst case
    rie_max = np.mean(np.exp(-bedroc_alpha * x_max)) / C
    rie_min = np.mean(np.exp(-bedroc_alpha * x_min)) / C

    # 4. Normalize the score
    return (rie - rie_min) / (rie_max - rie_min)

In [ ]:
# Calculate the enrichment factor
def calculate_ef(y_true, y_score, percentages=[0.5, 1.0, 2.0, 3.0,4.0, 5.0]):

    # y_true: true binary labels
    y_true = np.asarray(y_true)
    # y_score: predicted scores
    y_score = np.asarray(y_score)
    
    # Check the inputs
    if len(y_true) != len(y_score):
        raise ValueError("y_true and y_score must have the same length")
    
    N = len(y_true)  # Total number of compounds
    n_active = np.sum(y_true == 1)  # Total number of active compounds
    
    if n_active == 0:
        print("Warning: no active compounds were found; returning EF = 0")
        return {f"EF_{p}%": 0.0 for p in percentages}
    
    # Sort compounds by predicted score in descending order
    sorted_indices = np.argsort(y_score)[::-1]
    sorted_labels = y_true[sorted_indices]
    
    results = {}
    
    for percentage in percentages:
        # Calculate the number of compounds corresponding to the top x%
        n_selected = max(1, int(N * percentage / 100))  # Select at least one compound
        
        # Number of active compounds in the top x%
        n_active_in_selected = np.sum(sorted_labels[:n_selected])
        
        # Calculate the enrichment factor
        # EF = active-compound fraction in the top x% / active-compound fraction in the full dataset
        ef_value = (n_active_in_selected / n_selected) / (n_active / N)
        
        results[f"EF_{percentage}%"] = ef_value
        
        # Print detailed information
        print(f"EF_{percentage}%:")
        print(f"  - Selected compounds: {n_selected}/{N} ({percentage}%)")
        print(f"  - Active compounds among selected compounds: {n_active_in_selected}")
        print(f"  - Active-compound fraction: {n_active_in_selected/n_selected:.4f}")
        print(f"  - Overall active-compound fraction: {n_active/N:.4f}")
        print(f"  - EF value: {ef_value:.4f}")
        print()
    
    return results

In [ ]:
def objective(trial, fold_cache, bedroc_alpha):

    # ---------- 1. Search space ----------
    params = {
        "tree_method": "hist",     # GPU histogram algorithm
        "device": 'cuda',  # GPU prediction/training device
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "random_state": 42,

        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth":     trial.suggest_int("max_depth", 5, 10),
        "subsample":     trial.suggest_float("subsample", 0.3, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 0.8),
        "lambda": trial.suggest_float("lambda", 1e-3, 1e3, log=True),
        "alpha":  trial.suggest_float("alpha", 1e-3, 300, log=True),
        "gamma":  trial.suggest_float("gamma", 0, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 12),
        "scale_pos_weight": 14
    }

    # Folds come from FOLD_CACHE, i.e. the stratified split fixed in the cell above.
    # Every trial therefore sees byte-identical training and validation data.
    all_y_true = []
    all_y_pred = []
    best_iters = []  # Store the best iteration number from each fold
    fold_aucs = [] # record for mediumpruner

    for fold_id, (dtrain, dvalid, y_valid) in enumerate(fold_cache, start=1):

        # ---------- 4. Model training with early stopping ----------
        model = xgb.train(
            params,
            dtrain,
            num_boost_round=10000,
            evals=[(dvalid, "eval")],
            early_stopping_rounds=50,
            verbose_eval=False
        )
        best_iters.append(model.best_iteration + 1)  # Record the best iteration number; XGBoost uses 0-based indexing, so add 1

        # Predict probabilities
        y_pred = model.predict(dvalid, iteration_range=(0, model.best_iteration + 1))

        # Collect the results
        all_y_true.append(y_valid)
        all_y_pred.append(y_pred)

        # report per fold for pruner
        fold_aucs.append(roc_auc_score(y_valid, y_pred))
        trial.report(float(np.mean(fold_aucs)), step=fold_id)
        if trial.should_prune():
            del model
            gc.collect()  # a pruned trial would otherwise leave its booster on the GPU
            raise optuna.TrialPruned()

        del model  # free the booster before the next fold allocates its own

         # Calculate the average best iteration number
    avg_best_iter = int(np.round(np.mean(best_iters)))
    trial.set_user_attr('avg_best_iter', avg_best_iter)  # Save it as a user attribute

        # Concatenate predictions and labels
    y_true_concat = np.concatenate(all_y_true)
    y_pred_concat = np.concatenate(all_y_pred)

    # ---------- 5. Final evaluation ----------

    # Calculate BEDROC
    bedroc = calculate_bedroc(y_true_concat, y_pred_concat, bedroc_alpha=bedroc_alpha)
    auc = roc_auc_score(y_true_concat, y_pred_concat)
        # Save both metrics
    trial.set_user_attr("roc_auc", auc)
    trial.set_user_attr("bedroc", bedroc)

    print(f"[Validation] BEDROC alpha = {bedroc_alpha}")
    print(f"the best iteration number is {avg_best_iter}")
    return auc

In [ ]:
# Run Optuna Bayesian optimization
study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2), # prune the trial if not good
    sampler=optuna.samplers.TPESampler(seed=42) # random seed
    )
study.optimize(partial(objective, fold_cache=FOLD_CACHE, bedroc_alpha=80.5), n_trials=200, show_progress_bar=True)
best_params = study.best_params
best_iter = study.best_trial.user_attrs['avg_best_iter']
print("Best hyperparameters:", best_params)
print("Best iter:", best_iter)

In [ ]:
# The per-fold copies are no longer needed; free them before building the full-data matrix
del FOLD_CACHE
gc.collect()

final_params = {
    'objective': 'binary:logistic',  # Binary classification task
    'eval_metric': 'auc',       # Evaluation metric
    'tree_method': 'hist',
    'device': 'cuda',
    'scale_pos_weight':14, # balance active/inactive
    **best_params
}

Xy_train = xgb.QuantileDMatrix(X_train, label=y_train)
final_model = xgb.train(final_params, Xy_train, num_boost_round=best_iter)

# Use the same path for saving and evaluation.
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
final_model.save_model(MODEL_PATH)
print("saved:", os.path.abspath(MODEL_PATH))

In [ ]:
del Xy_train, X_train, y_train, FOLDS, final_model
gc.collect()


In [ ]:
# Define a function to test the trained model
def run_the_model_full(Xy,y,kinase_name):

    # Load the model
    final_model = xgb.Booster()
    final_model.load_model(MODEL_PATH)
    y_pred_prob = final_model.predict(Xy)  # Predicted probabilities

    # Automatically select the optimal threshold
    precisions, recalls, thresholds = precision_recall_curve(y, y_pred_prob)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    optimal_idx = np.argmax(f1_scores)
    y_pred = (y_pred_prob > thresholds[optimal_idx])

    # Calculate common evaluation metrics
    auc = roc_auc_score(y, y_pred_prob)
    auc_var = np.var(y_pred_prob) / len(y_pred_prob)
    z = norm.ppf(1 - (1 - 0.95) / 2)  # Calculate the z value
    lower_bound = auc - z * np.sqrt(auc_var)
    upper_bound = auc + z * np.sqrt(auc_var)

    acc = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    mcc = matthews_corrcoef(y, y_pred)
    logloss = log_loss(y, y_pred_prob)

    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0.0
    npv = tn / (tn + fn) if (tn + fn) != 0 else 0.0

     # Calculate confidence intervals by bootstrapping
    n_bootstraps = 1000
    bootstrapped_aucs = []

    # Convert y and y_pred_prob to NumPy arrays
    y = np.array(y)
    y_pred_prob = np.array(y_pred_prob)

    rng = np.random.RandomState(42)
    for _ in range(n_bootstraps):
        indices = rng.choice(len(y), size=len(y), replace=True)
        auc_bootstrap = roc_auc_score(y[indices], y_pred_prob[indices])
        bootstrapped_aucs.append(auc_bootstrap)
    ci_lower = np.percentile(bootstrapped_aucs, 2.5)
    ci_upper = np.percentile(bootstrapped_aucs, 97.5)
    print(f"AUC: {auc:.3f} (95% CI: [{ci_lower:.3f}, {ci_upper:.3f}])")

    # Calculate BEDROC
    bedroc_score = calculate_bedroc(y, y_pred_prob, bedroc_alpha=80.5)
    ef_score = calculate_ef(y, y_pred_prob)
    # Print evaluation results
    print(f"Best threshold: {thresholds[optimal_idx]:.4f}")
    print(f"AUC: {auc:.4f} (95% CI: {lower_bound:.3f}, {upper_bound:.3f})")
    print(f"bedroc: {bedroc_score:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision(PPV): {precision:.4f}")
    print(f"Recall(Sensitivity): {recall:.4f}")
    print(f"F1 score: {f1:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"NPV: {npv:.4f}")
    print(f"MCC: {mcc:.4f}")
    print(f"Log Loss: {logloss:.4f}")
    print(confusion_matrix(y, y_pred))
    print(f"True Negative = {tn}")
    print(f"True Positive = {tp}")
    print(f"False Negative = {fn}")
    print(f"False Positive = {fp}")
    print("Class distribution:", Counter(y))
    print("Positive-class fraction:", np.mean(y))

    if kinase_name is not None:
        df_output = pd.DataFrame({
            "Best threshold": f"{thresholds[optimal_idx]:.4f}",
            "AUC": f"{auc:.4f}",
            "AUC 95% CI lower_bound": f"{lower_bound:.3f}",
            "AUC 95% CI upper_bound": f"{upper_bound:.3f}",
            "bedroc": f"{bedroc_score:.4f}",
            "Accuracy": f"{acc:.4f}",
            "Precision(PPV)": f"{precision:.4f}",
            "Recall(Sensitivity)": f"{recall:.4f}",
            "F1 score": f"{f1:.4f}",
            "Specificity": f"{specificity:.4f}",
            "NPV": f"{npv:.4f}",
            "MCC": f"{mcc:.4f}",
            "Log Loss": f"{logloss:.4f}",
            "True Negative": f"{tn}",
            "True Positive": f"{tp}",
            "False Negative": f"{fn}",
            "False Positive": f"{fp}"
        },index=[kinase_name])
        return bedroc_score,df_output,ef_score
    else:
        return bedroc_score

In [ ]:
# vsds test dataset
vsds = pd.read_csv(_kp('dataset/VSDS_truedecoy_integrated_data_with_protein_seq.csv'),
    usecols=["KLIFS", "molecule_label"],
)

# This file carries a header row of column indices, hence skiprows=1 / header=None.
vsds_mol_fp = pd.read_csv(_kp('dataset/VSDS_truedecoy_LSAE_latent_top95_features_v4.csv'),
    skiprows=1, header=None, dtype=np.float32,
)

vsds_bio = vsds["molecule_label"]

# Same helper as the training set, so the column order is guaranteed to match the model.
X_vsds = build_features(vsds, vsds_mol_fp, KLIFS_file, KLIFS_sae_top)

print("vsds mol latent :", vsds_mol_fp.shape)
print("X_vsds          :", X_vsds.shape, X_vsds.dtype)

assert X_vsds.shape[1] == N_FEATURES, "VSDS features must match the training columns"
vsds_Xy = xgb.DMatrix(X_vsds, label=vsds_bio)
del X_vsds, vsds_mol_fp, vsds
gc.collect()

In [ ]:
print('evaluation metrics on VSDS test set:')
vsds_test = run_the_model_full(vsds_Xy,vsds_bio,None)